# Turniting AI Detector — GPU training (Colab)

Full transformer recipe (`configs/training_gpu.yaml`: mDeBERTa-v3-base, 512 tokens, stride 128, all layers,
AMP, ~150k windows, 3 epochs) on `TURNITING_DATASET_V1`.

Code comes from the public repo (branch `exp/ai-detector-training`); `MyDrive/turniting/workdir/` keeps the
dataset, checkpoints, models, reports and logs (persistent). Reference manifests and the frozen evaluation ids
of the CPU run are versioned in `training/datasets/manifests/`.
The dataset is rebuilt from the official public sources with the same scripts and seed, and checked against
the reference split counts. Re-running all cells resumes from `last` after a disconnect.

Independent detector — not a Turnitin model. No Turnitin reports or private documents are used.

In [16]:
!git pull -q origin exp/ai-detector-training; python scripts/summarize_gpu_run.py --out-dir /content/drive/MyDrive/turniting/workdir/reports | tail -n 3

GPU_SUMMARY_JSON_START
{"generatedAt":"2026-10-17T02:18:50Z","datasetVersion":"TURNITING_DATASET_V1","experiments":{"20261017_0117_mdeberta_gpu_es_acad_v1":{"mode":"refine","model":"microsoft/mdeberta-v3-base","device":"cuda","gpu":"NVIDIA A100-SXM4-40GB","vram_gb":42.41,"cuda":"13.0","train_windows":30000,"epochs":1,"total_steps":938,"steps_per_epoch":938,"max_length":512,"stride":128,"trainable_params":278810882,"batch":[32,1],"learning_rate":1e-05,"stopped_at_step":938,"early_stopped":false,"train_seconds":502.9,"history":[{"step":938,"epoch":1.0,"train_loss":0.1569,"val_loss":0.4189,"val_macro_f1":0.8637,"val_fpr":0.2636,"gpu_mem_gb":25.535}],"best_aggregation":"mean","overfitting":{"train_loss":0.4203,"val_loss":0.4308,"train_macro_f1":0.8575,"val_macro_f1":0.8542,"gap_macro_f1":0.0033},"checkpoint_bytes":{"best":1135930811,"last":3366649531},"sample_audit":{"training_windows":30000,"training_docs":23979,"machine_window_share":0.5468333333333333,"spanish_windows":12230,"academic_w

In [17]:
!cd /content/drive/MyDrive/turniting/workdir && du -sh checkpoints/*gpu*/best checkpoints/*gpu*/last models/gpu/* && ls models/gpu/calibrated_spanish_academic && ls reports/eval | grep -c GPU

1.1G	checkpoints/20261017_0117_mdeberta_gpu_es_acad_v1/best
1.1G	checkpoints/20261017_0117_mdeberta_gpu_v1/best
3.2G	checkpoints/20261017_0117_mdeberta_gpu_es_acad_v1/last
3.2G	checkpoints/20261017_0117_mdeberta_gpu_v1/last
1.1G	models/gpu/calibrated_spanish_academic
1.1G	models/gpu/global
1.1G	models/gpu/spanish_academic
added_tokens.json      model_manifest.json	thresholds.json
calibration.json       model.safetensors	tokenizer_config.json
config.json	       special_tokens_map.json	tokenizer.json
inference_config.json  spm.model
18


In [2]:
# 1. Hardware audit
!nvidia-smi
import torch, shutil, psutil
print('torch', torch.__version__, '| cuda available', torch.cuda.is_available(), '| cuda', torch.version.cuda)
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print('GPU', p.name, f'{p.total_memory/1e9:.1f} GB', '| bf16', torch.cuda.is_bf16_supported())
print(f'RAM {psutil.virtual_memory().total/1e9:.1f} GB | disk free {shutil.disk_usage("/content").free/1e9:.1f} GB')
assert torch.cuda.is_available(), 'Runtime is CPU: switch to a GPU runtime before training'

Sat Oct 17 00:21:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             47W /  400W |       6MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
# 2. Drive
from google.colab import drive
drive.mount('/content/drive')
import os, json, shutil
DRIVE = '/content/drive/MyDrive/turniting'
W = f'{DRIVE}/workdir'
for sub in ['checkpoints', 'models/baseline', 'reports/eval', 'logs', 'datasets']:
    os.makedirs(f'{W}/{sub}', exist_ok=True)
print(sorted(os.listdir(DRIVE)))

Mounted at /content/drive
['workdir']


In [4]:
# 3. Code + dependencies
if not os.path.exists('/content/Turniting/.git'):
    !git clone -q --branch exp/ai-detector-training https://github.com/Anthgg/Turniting.git /content/Turniting
else:
    !git -C /content/Turniting fetch -q origin exp/ai-detector-training && git -C /content/Turniting checkout -q exp/ai-detector-training && git -C /content/Turniting pull -q --ff-only
%cd /content/Turniting/training
!pip -q install transformers==4.46.3 accelerate==1.1.1 sentencepiece==0.2.0 langid==1.1.6 datasketch==1.6.5 tiktoken protobuf
!git log --oneline -1

/content/Turniting/training
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 45.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 96.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 108.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.2/89.2 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 130.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingfa

In [5]:
# 4. Dataset: rebuild TURNITING_DATASET_V1 from the official sources once; persist master to Drive
!mkdir -p /content/ref && cp datasets/manifests/split_manifest.json datasets/manifests/master_manifest.json datasets/manifests/eval_sets.parquet datasets/manifests/spanish_weight_selection.json /content/ref/
if not os.path.exists(f'{W}/datasets/normalized/master/train'):
    !python scripts/download_datasets.py 2>&1 | grep -E '==|->' | tail -n 30
    !python scripts/build_dataset.py 2>&1 | grep -E 'stage1 |exact|near|groups|leakage|done|Error|Traceback' | tail -n 40
    shutil.copytree('datasets/normalized/master', f'{W}/datasets/normalized/master', dirs_exist_ok=True)
    for f in ['split_manifest.parquet', 'index.parquet', 'stage2_summary.json']:
        shutil.copy2(f'datasets/normalized/{f}', f'{W}/datasets/normalized/{f}')
    shutil.copytree('datasets/manifests', f'{W}/datasets/manifests', dirs_exist_ok=True)
    shutil.copytree('reports', f'{W}/reports/rebuild', dirs_exist_ok=True)
!rm -rf datasets/normalized && ln -s {W}/datasets/normalized datasets/normalized
!cp {W}/datasets/manifests/*.json datasets/manifests/ && ls datasets/normalized/master

== iberautextification (hf)
   -> DOWNLOADED 
== autextification2023 (hf)
   -> DOWNLOADED 
== multitude (zenodo)
   -> BLOCKED_ACCESS Zenodo record 10013755 access_right=restricted; files require owner approval
== semeval2024_task8 (zenodo)
   -> DOWNLOADED 
== m4 (github_raw)
   -> DOWNLOADED 
== mage (hf)
   -> DOWNLOADED 
== hc3 (hf)
   -> DOWNLOADED 
== hc3_chinese (hf)
   -> DOWNLOADED 
== raid (hf)
   -> DOWNLOADED 
== turingbench (hf)
   -> DOWNLOADED 
== ghostbuster_essay (github_tree)
   -> DOWNLOADED 
00:33:14 stage1 iberautextification: {'rows_original': 153028, 'invalid_broken_html': 4, 'rows_valid': 153024}
00:33:42 stage1 autextification2023: {'rows_original': 107868, 'invalid_below_min_chars': 92, 'invalid_below_min_words': 4, 'invalid_broken_html': 3, 'rows_valid': 107769}
00:50:29 stage1 semeval2024_task8: {'rows_original': 485128, 'invalid_broken_html': 19, 'invalid_below_min_chars': 51, 'invalid_below_min_words': 10, 'invalid_empty_or_whitespace': 1, 'rows_valid': 4

In [6]:
# 5. Verify the rebuild matches the CPU-run reference
ref = json.load(open('/content/ref/split_manifest.json'))
new = json.load(open('datasets/manifests/split_manifest.json'))
print('reference splits:', ref['split_counts'])
print('rebuilt splits  :', new['split_counts'])
print('reference status:', ref['status_counts'])
print('rebuilt status  :', new['status_counts'])
SAME = ref['split_counts'] == new['split_counts'] and ref['status_counts'] == new['status_counts']
print('IDENTICAL SPLITS' if SAME else 'WARNING: splits differ from the CPU run -> eval sets rebuilt from this master')

reference splits: {'train': 865590, 'test': 201824, 'validation': 93894, 'unseen_domain_test': 49916, 'ood_test': 44406, 'unseen_generator_test': 36468, 'adversarial_test': 16276, 'mixed': 15118, 'adversarial_validation': 14765}
rebuilt splits  : {'train': 865590, 'test': 201824, 'validation': 93894, 'unseen_domain_test': 49916, 'ood_test': 44406, 'unseen_generator_test': 36468, 'adversarial_test': 16276, 'mixed': 15118, 'adversarial_validation': 14765}
reference status: {'KEPT': 1338257, 'EXACT_DUPLICATE': 625144, 'HOLDOUT_GENERATOR_IN_TRAIN_GROUP': 113331, 'HOLDOUT_DOMAIN_IN_TRAIN_GROUP': 102591, 'NEAR_DUPLICATE': 15795, 'LABEL_CONFLICT': 10}
rebuilt status  : {'KEPT': 1338257, 'EXACT_DUPLICATE': 625144, 'HOLDOUT_GENERATOR_IN_TRAIN_GROUP': 113331, 'HOLDOUT_DOMAIN_IN_TRAIN_GROUP': 102591, 'NEAR_DUPLICATE': 15795, 'LABEL_CONFLICT': 10}
IDENTICAL SPLITS


In [7]:
# 6. Persistent models/checkpoints + frozen evaluation ids
!rm -rf checkpoints models && ln -s {W}/checkpoints checkpoints && ln -s {W}/models models
!cp -n /content/ref/spanish_weight_selection.json models/baseline/spanish_weight_selection.json
if SAME and not os.path.exists('models/eval_sets.parquet'):
    shutil.copy2('/content/ref/eval_sets.parquet', 'models/eval_sets.parquet')  # same rows as the CPU run
import pandas as pd
if os.path.exists('models/eval_sets.parquet'):
    print(pd.read_parquet('models/eval_sets.parquet').eval_set.value_counts())

cp: warning: behavior of -n is non-portable and may change in future; use --update=none instead
eval_set
validation_large               40000
test_large                     40000
test                            4000
adversarial                     2400
unseen_generator                2400
ood                             1500
test_spanish                    1200
unseen_domain                   1200
test_academic                   1000
validation                       900
adversarial_clean_reference      600
validation_academic              450
validation_spanish               450
Name: count, dtype: int64


In [8]:
# 7. Experiment ids (persisted for resume)
import time
ids_file = f'{W}/experiment_ids.json'
if os.path.exists(ids_file):
    ids = json.load(open(ids_file))
else:
    stamp = time.strftime('%Y%m%d_%H%M')
    ids = {'global': f'{stamp}_mdeberta_gpu_v1', 'refine': f'{stamp}_mdeberta_gpu_es_acad_v1'}
    json.dump(ids, open(ids_file, 'w'))
print(ids)

{'global': '20261017_0117_mdeberta_gpu_v1', 'refine': '20261017_0117_mdeberta_gpu_es_acad_v1'}


In [9]:
# 8. Sampling audit BEFORE training (aborts if RAID / Iber / AuTex are missing)
!python scripts/train_transformer.py --config configs/training_gpu.yaml --mode global --dry-run-sample --experiment-id dryrun_gpu 2>&1 | grep -E 'sample audit|dry run|ABORT|Error|Traceback'
!cp checkpoints/dryrun_gpu/sample_audit.json {W}/reports/sample_audit_global.json

01:21:15 sample audit: {"training_windows": 148344, "training_docs": 120000, "machine_window_share": 0.5420306854338565, "spanish_windows": 29215, "academic_windows": 21332, "adversarial_windows": 15763, "by_dataset": {"autextification2023": {"docs": 25405, "windows": 25405, "machine_docs": 12828}, "ghostbuster_essay": {"docs": 4625, "windows": 8930, "machine_docs": 3827}, "hc3": {"docs": 7249, "windows": 7562, "machine_docs": 3521}, "hc3_chinese": {"docs": 3586, "windows": 3599, "machine_docs": 1793}, "iberautextification": {"docs": 25315, "windows": 34159, "machine_docs": 12079}, "m4": {"docs": 10106, "windows": 12365, "machine_docs": 5682}, "mage": {"docs": 12898, "windows": 15583, "machine_docs": 6613}, "raid": {"docs": 12472, "windows": 17066, "machine_docs": 8237}, "semeval2024_task8": {"docs": 13713, "windows": 18716, "machine_docs": 6847}, "turingbench": {"docs": 4631, "windows": 4959, "machine_docs": 2508}}}
01:21:15 dry run OK: no training performed


In [10]:
# 9. Training helper: streams to a Drive log, resumes from last/ after a disconnect
import re, subprocess
SHOW = re.compile(r'EVAL|done:|early stopping|Error|Traceback|ABORT|sample audit|resum|step (\d+)00/')
def train(mode, exp_id, extra=()):
    exp = f'checkpoints/{exp_id}'
    if os.path.exists(f'{exp}/experiment.json'):
        print(exp_id, 'already finished'); return
    args = ['python', 'scripts/train_transformer.py', '--config', 'configs/training_gpu.yaml', '--mode', mode]
    if os.path.exists(f'{exp}/last/trainer_state.pt'):
        args += ['--resume-from-checkpoint', f'{exp}/last']
    else:
        args += ['--experiment-id', exp_id, *extra]
    log = f'{W}/logs/{exp_id}.log'
    print(' '.join(args), '->', log)
    with open(log, 'a') as fh:
        proc = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            fh.write(line); fh.flush()
            if SHOW.search(line):
                print(line, end='', flush=True)
    assert proc.wait() == 0, f'training failed, see {log}'

In [11]:
# 10. GLOBAL_MULTILINGUAL_TRANSFORMER
train('global', ids['global'])

python scripts/train_transformer.py --config configs/training_gpu.yaml --mode global --experiment-id 20261017_0117_mdeberta_gpu_v1 -> /content/drive/MyDrive/turniting/workdir/logs/20261017_0117_mdeberta_gpu_v1.log
01:24:30 sample audit: {"training_windows": 148344, "training_docs": 120000, "machine_window_share": 0.5420306854338565, "spanish_windows": 29215, "academic_windows": 21332, "adversarial_windows": 15763, "by_dataset": {"autextification2023": {"docs": 25405, "windows": 25405, "machine_docs": 12828}, "ghostbuster_essay": {"docs": 4625, "windows": 8930, "machine_docs": 3827}, "hc3": {"docs": 7249, "windows": 7562, "machine_docs": 3521}, "hc3_chinese": {"docs": 3586, "windows": 3599, "machine_docs": 1793}, "iberautextification": {"docs": 25315, "windows": 34159, "machine_docs": 12079}, "m4": {"docs": 10106, "windows": 12365, "machine_docs": 5682}, "mage": {"docs": 12898, "windows": 15583, "machine_docs": 6613}, "raid": {"docs": 12472, "windows": 17066, "machine_docs": 8237}, "sem

In [12]:
# 11. SPANISH_ACADEMIC_TRANSFORMER (from global best; 70 % Spanish/academic + 30 % replay; lr 1e-5)
train('refine', ids['refine'], ['--init-from', f"checkpoints/{ids['global']}/best"])

python scripts/train_transformer.py --config configs/training_gpu.yaml --mode refine --experiment-id 20261017_0117_mdeberta_gpu_es_acad_v1 --init-from checkpoints/20261017_0117_mdeberta_gpu_v1/best -> /content/drive/MyDrive/turniting/workdir/logs/20261017_0117_mdeberta_gpu_es_acad_v1.log
02:01:02 sample audit: {"training_windows": 30000, "training_docs": 23979, "machine_window_share": 0.5468333333333333, "spanish_windows": 12230, "academic_windows": 8845, "adversarial_windows": 2474, "by_dataset": {"autextification2023": {"docs": 7796, "windows": 7796, "machine_docs": 3945}, "ghostbuster_essay": {"docs": 2516, "windows": 4768, "machine_docs": 1764}, "hc3": {"docs": 577, "windows": 599, "machine_docs": 289}, "hc3_chinese": {"docs": 299, "windows": 300, "machine_docs": 161}, "iberautextification": {"docs": 5226, "windows": 7288, "machine_docs": 2414}, "m4": {"docs": 1615, "windows": 1860, "machine_docs": 1020}, "mage": {"docs": 1412, "windows": 1669, "machine_docs": 729}, "raid": {"docs"

In [13]:
# 12. Package + calibrate (VALIDATION only)
g, r = ids['global'], ids['refine']
!python scripts/calibrate.py --experiment checkpoints/{g} --out models/gpu/global --name GLOBAL_MULTILINGUAL_TRANSFORMER_GPU
!python scripts/calibrate.py --experiment checkpoints/{r} --out models/gpu/spanish_academic --name SPANISH_ACADEMIC_TRANSFORMER_GPU
!python scripts/calibrate.py --experiment checkpoints/{r} --out models/gpu/calibrated_spanish_academic --name CALIBRATED_SPANISH_ACADEMIC_TRANSFORMER_GPU --calibrate

packaged GLOBAL_MULTILINGUAL_TRANSFORMER_GPU -> models/gpu/global thresholds={'BALANCED': 0.93, 'LOW_FALSE_POSITIVE': 0.99, 'HIGH_PRECISION_AI': 0.9650000000000001}
packaged SPANISH_ACADEMIC_TRANSFORMER_GPU -> models/gpu/spanish_academic thresholds={'BALANCED': 0.985, 'LOW_FALSE_POSITIVE': 0.99, 'HIGH_PRECISION_AI': 0.99}
packaged CALIBRATED_SPANISH_ACADEMIC_TRANSFORMER_GPU -> models/gpu/calibrated_spanish_academic thresholds={'BALANCED': 0.505, 'LOW_FALSE_POSITIVE': 0.875, 'HIGH_PRECISION_AI': 0.5850000000000001}
calibration: isotonic {'ece_before': 0.13060672086399416, 'ece_after': 2.90733960287977e-17, 'brier_before': 0.11298423192036292, 'brier_after': 0.053263174510291264}


In [14]:
# 13. VALIDATION only (model choice)
GPU_MODELS = [('GLOBAL_MULTILINGUAL_TRANSFORMER_GPU', 'models/gpu/global'),
              ('SPANISH_ACADEMIC_TRANSFORMER_GPU', 'models/gpu/spanish_academic'),
              ('CALIBRATED_SPANISH_ACADEMIC_TRANSFORMER_GPU', 'models/gpu/calibrated_spanish_academic')]
for name, path in GPU_MODELS:
    !python scripts/evaluate.py --model {path} --name {name}__validation 2>&1 | grep -E 'macroF1|Error|Traceback'
!cp -r reports/eval/* {W}/reports/eval/

02:10:34 GLOBAL_MULTILINGUAL_TRANSFORMER_GPU__validation validation: n=900 macroF1=0.9221837944664031 f1=0.9239130434782609 fpr=0.07954545454545454
02:10:35 GLOBAL_MULTILINGUAL_TRANSFORMER_GPU__validation validation_spanish: n=450 macroF1=0.9244205576083306 f1=0.9230769230769231 fpr=0.04072398190045249
02:10:37 GLOBAL_MULTILINGUAL_TRANSFORMER_GPU__validation validation_academic: n=450 macroF1=0.9238427078148332 f1=0.9306122448979591 fpr=0.07389162561576355
02:10:55 SPANISH_ACADEMIC_TRANSFORMER_GPU__validation validation: n=900 macroF1=0.9222033973652883 f1=0.9234135667396062 fpr=0.07272727272727272
02:10:57 SPANISH_ACADEMIC_TRANSFORMER_GPU__validation validation_spanish: n=450 macroF1=0.9177578772147335 f1=0.9164785553047404 fpr=0.049773755656108594
02:10:59 SPANISH_ACADEMIC_TRANSFORMER_GPU__validation validation_academic: n=450 macroF1=0.934961699667582 f1=0.9411764705882353 fpr=0.06896551724137931
02:11:17 CALIBRATED_SPANISH_ACADEMIC_TRANSFORMER_GPU__validation validation: n=900 macr

In [15]:
# 14. TEST-side sets, once, after the validation choice
for name, path in GPU_MODELS:
    !python scripts/evaluate.py --model {path} --name {name} --final 2>&1 | grep -E 'macroF1|Error|Traceback'
!cp -r reports/eval/* {W}/reports/eval/ && cp checkpoints/*/experiment.json checkpoints/*/train_log.jsonl {W}/reports/ 2>/dev/null
!for d in checkpoints/*/; do n=$(basename $d); cp $d/experiment.json {W}/reports/$n.experiment.json 2>/dev/null; cp $d/train_log.jsonl {W}/reports/$n.train_log.jsonl 2>/dev/null; cp $d/sample_audit.json {W}/reports/$n.sample_audit.json 2>/dev/null; done
!ls {W}/reports {W}/reports/eval

02:11:36 GLOBAL_MULTILINGUAL_TRANSFORMER_GPU validation: n=900 macroF1=0.9221837944664031 f1=0.9239130434782609 fpr=0.07954545454545454
02:11:36 GLOBAL_MULTILINGUAL_TRANSFORMER_GPU validation_spanish: n=450 macroF1=0.9244205576083306 f1=0.9230769230769231 fpr=0.04072398190045249
02:11:37 GLOBAL_MULTILINGUAL_TRANSFORMER_GPU validation_academic: n=450 macroF1=0.9238427078148332 f1=0.9306122448979591 fpr=0.07389162561576355
02:11:53 GLOBAL_MULTILINGUAL_TRANSFORMER_GPU test: n=4000 macroF1=0.8944755169380614 f1=0.9019149929939281 fpr=0.12797461660497092
02:11:56 GLOBAL_MULTILINGUAL_TRANSFORMER_GPU test_spanish: n=1200 macroF1=0.869998555539506 f1=0.8704318936877077 fpr=0.15309446254071662
02:12:00 GLOBAL_MULTILINGUAL_TRANSFORMER_GPU test_academic: n=1000 macroF1=0.887637090481006 f1=0.907075873827792 fpr=0.11793611793611794
02:12:09 GLOBAL_MULTILINGUAL_TRANSFORMER_GPU ood: n=1500 macroF1=0.8854274499654935 f1=0.8973429951690821 fpr=0.21733333333333332
02:12:25 GLOBAL_MULTILINGUAL_TRANSFORM